<a href="https://colab.research.google.com/github/simasaadi/toronto-water-analytics/blob/main/notebooks/00_load_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

# -------------------------------------------------------------
# 1. Load CSV directly from your GitHub repo (RAW link)
# -------------------------------------------------------------
csv_url = "https://raw.githubusercontent.com/simasaadi/toronto-water-analytics/refs/heads/main/data/monthly_overall_stats.csv"

monthly_df = pd.read_csv(csv_url)
print("Loaded rows:", len(monthly_df))
print("Columns:", list(monthly_df.columns))
display(monthly_df.head())

# -------------------------------------------------------------
# 2. Create a small SQLite database in Colab
# -------------------------------------------------------------
db_path = "toronto_water_demo.db"
conn = sqlite3.connect(db_path)

# Write the dataframe as a SQL table
monthly_df.to_sql("monthly_overall_stats", conn, if_exists="replace", index=False)

print("Database created at:", db_path)

# -------------------------------------------------------------
# 3. Run a simple SQL query to check everything works
#    (note: using actual column names: year, month_name, mean)
# -------------------------------------------------------------
query = """
SELECT
    year,
    month_name,
    mean
FROM monthly_overall_stats
ORDER BY year, month;
"""

result = pd.read_sql(query, conn)
display(result.head(10))

conn.close()


Loaded rows: 712
Columns: ['activity_datetime', 'count', 'mean', 'median', 'min', 'max', 'year', 'month', 'month_name']


,activity_datetime,count,mean,median,min,max,year,month,month_name
0,1964-10-31,72,82.542500,9.50,0.01,964.0,1964,10,October
1,1964-11-30,76,402.848158,6.35,0.01,11634.0,1964,11,November
2,1964-12-31,87,149.158506,8.40,0.01,2778.0,1964,12,December
3,1965-01-31,108,646.370093,10.30,0.05,30654.0,1965,1,January
4,1965-02-28,58,170.560345,6.35,0.01,1790.0,1965,2,February


Database created at: toronto_water_demo.db


,year,month_name,mean
0,1964,October,82.542500
1,1964,November,402.848158
2,1964,December,149.158506
3,1965,January,646.370093
4,1965,February,170.560345
5,1965,March,194.895766
6,1965,April,118.922500
7,1965,May,139.249231
8,1965,June,518.081765
9,1965,July,361.164221


In [ ]:
import sqlite3
import pandas as pd

# -------------------------------------------------------------
# 1. Read all four curated CSVs from GitHub (RAW links)
#    These are in: data/   in your repo
# -------------------------------------------------------------
base = "https://raw.githubusercontent.com/simasaadi/toronto-water-analytics/refs/heads/main/data/"

files = {
    "monthly_overall_stats": "monthly_overall_stats.csv",
    "monthly_top_characteristics_stats": "monthly_top_characteristics_stats.csv",
    "location_summary_stats": "location_summary_stats.csv",
    "seasonal_median_by_month": "seasonal_median_by_month.csv",
}

tables = {}

for table_name, fname in files.items():
    url = base + fname
    df = pd.read_csv(url)
    print(f"Loaded {table_name}: {len(df)} rows")
    print("    Columns:", list(df.columns))
    tables[table_name] = df

# -------------------------------------------------------------
# 2. Create the project database with all four tables
# -------------------------------------------------------------
db_path = "toronto_water.db"
conn = sqlite3.connect(db_path)

for table_name, df in tables.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"  -> wrote table '{table_name}'")

print("Database created at:", db_path)

# -------------------------------------------------------------
# 3. Quick test query from monthly_overall_stats
# -------------------------------------------------------------
test_query = """
SELECT
    year,
    month_name,
    mean
FROM monthly_overall_stats
ORDER BY year, month
LIMIT 5;
"""
test_result = pd.read_sql(test_query, conn)
print("\nSample from monthly_overall_stats:")
display(test_result)

conn.close()


Loaded monthly_overall_stats: 712 rows
    Columns: ['activity_datetime', 'count', 'mean', 'median', 'min', 'max', 'year', 'month', 'month_name']
Loaded monthly_top_characteristics_stats: 4140 rows
    Columns: ['activity_datetime', 'characteristicname', 'count', 'mean', 'median', 'min', 'max', 'year', 'month', 'month_name']
Loaded location_summary_stats: 648 rows
    Columns: ['monitoringlocationid', 'monitoringlocationname', 'monitoringlocationlatitude', 'monitoringlocationlongitude', 'count', 'mean', 'median', 'min', 'max']
Loaded seasonal_median_by_month: 12 rows
    Columns: ['month', 'median_resultvalue', 'month_name']
  -> wrote table 'monthly_overall_stats'
  -> wrote table 'monthly_top_characteristics_stats'
  -> wrote table 'location_summary_stats'
  -> wrote table 'seasonal_median_by_month'
Database created at: toronto_water.db

Sample from monthly_overall_stats:


,year,month_name,mean
0,1964,October,82.542500
1,1964,November,402.848158
2,1964,December,149.158506
3,1965,January,646.370093
4,1965,February,170.560345
